<a href="https://colab.research.google.com/github/varshayinithangaraju30-lgtm/customer-segmentation-kmeans/blob/main/performance_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import time
import shutil
import numpy as np
import pandas as pd
from joblib import Parallel, delayed, Memory
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

URL = "https://raw.githubusercontent.com/varshayinithangaraju30-lgtm/customer-segmentation-kmeans/main/Mall_Customers.csv"
FEATURES = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]

base = pd.read_csv(URL).drop_duplicates().dropna(subset=FEATURES)


def make_data(n_rows):
    """Bigger dataset by resampling rows and adding small noise."""
    rng = np.random.default_rng(42)
    data = base[FEATURES].sample(n_rows, replace=True, random_state=42).reset_index(drop=True)
    return StandardScaler().fit_transform(data + rng.normal(0, 1, size=data.shape))


X = make_data(5000)
K_VALUES = list(range(2, 11))


def score_for_k(X, k):
    labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X)
    return k, round(silhouette_score(X, labels), 4)


# ---------- 1. Sequential ----------
def run_sequential(X, ks):
    return [score_for_k(X, k) for k in ks]


# ---------- 2. Parallel (all CPU cores) ----------
def run_parallel(X, ks, n_jobs=-1):
    if not ks:
        raise ValueError("k list is empty")
    return Parallel(n_jobs=n_jobs)(delayed(score_for_k)(X, k) for k in ks)


# ---------- 3. Caching ----------
CACHE_DIR = "cache_dir"
shutil.rmtree(CACHE_DIR, ignore_errors=True)
memory = Memory(CACHE_DIR, verbose=0)
cached_score_for_k = memory.cache(score_for_k)


def run_cached(X, ks):
    return [cached_score_for_k(X, k) for k in ks]


def timed(fn, *args):
    t0 = time.perf_counter()
    result = fn(*args)
    return result, time.perf_counter() - t0


seq_res, t_seq = timed(run_sequential, X, K_VALUES)
par_res, t_par = timed(run_parallel, X, K_VALUES)
_, t_cache1 = timed(run_cached, X, K_VALUES)
cache_res, t_cache2 = timed(run_cached, X, K_VALUES)

print("CPU cores available:", os.cpu_count())
summary = pd.DataFrame({
    "Method": ["Sequential", "Parallel", "Cached (1st run)", "Cached (2nd run)"],
    "Time (s)": [round(t_seq, 3), round(t_par, 3), round(t_cache1, 3), round(t_cache2, 3)],
})
summary["Speedup vs sequential"] = (t_seq / summary["Time (s)"]).round(2)
print(summary.to_string(index=False))

best_k = max(par_res, key=lambda r: r[1])
print("\nBest K by silhouette:", best_k)


# ---------- Tests ----------
def test_parallel_same_as_sequential():
    assert sorted(seq_res) == sorted(par_res)

def test_cached_same_as_sequential():
    assert sorted(seq_res) == sorted(cache_res)

def test_cache_second_run_faster():
    assert t_cache2 < t_cache1

def test_empty_k_list_error():
    try:
        run_parallel(X, [])
        assert False
    except ValueError:
        pass

for t in [test_parallel_same_as_sequential, test_cached_same_as_sequential,
          test_cache_second_run_faster, test_empty_k_list_error]:
    try:
        t()
        print("PASS:", t.__name__)
    except AssertionError:
        print("FAIL:", t.__name__)

CPU cores available: 2
          Method  Time (s)  Speedup vs sequential
      Sequential     9.797                   1.00
        Parallel     8.001                   1.22
Cached (1st run)     4.521                   2.17
Cached (2nd run)     0.007                1399.55

Best K by silhouette: (10, np.float64(0.468))
PASS: test_parallel_same_as_sequential
PASS: test_cached_same_as_sequential
PASS: test_cache_second_run_faster
PASS: test_empty_k_list_error
